In this notebook our goal is to create a pretrained model based on each of these architectures:
- Xception
- VGG16 and VGG19
- ResNet
- Inception
- MobileNet
- DenseNet
- NASNet
- EfficientNet
- ConvNext

We should read the papers of each of these architectures, identify what are the breakthroughs of each model and what is the thinking behind each of them.

https://keras.io/guides/transfer_learning/

## Imports and config

In [2]:
import numpy as np
import keras
from keras import layers
import tensorflow as tf
import matplotlib.pyplot as plt

import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # Suppress TensorFlow warnings

2026-04-15 16:20:30.368226: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776266430.394156  236607 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776266430.404063  236607 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776266430.500663  236607 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776266430.500716  236607 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776266430.500721  236607 computation_placer.cc:177] computation placer alr

Let's check if our GPU is recognized and set the VRAM limit for tensorflow to reserve

In [3]:
gpus = tf.config.list_physical_devices('GPU')
gpus

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

In [4]:
# 16GB Tesla T4 (Google Colab)
# 4GB RTX 3050 Ti Laptop (Afonso)
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)
    tf.config.set_logical_device_configuration(
        gpu,
        [tf.config.LogicalDeviceConfiguration(memory_limit=round(3.8*1024))]
    )

keras.mixed_precision.set_global_policy('mixed_float16')

## Data import

In [5]:
DATA_PATH = "../data"

image_size = (512, 512)
# image_size = (224, 224)
# input_shape = image_size + (3,)
batch_size = 4
n_classes = 23

train_ds, val_ds = keras.utils.image_dataset_from_directory(
    DATA_PATH,
    image_size=image_size,
    batch_size=batch_size,
    seed=255, # for reproducibility
    validation_split=0.2,
    subset="both",
    label_mode="categorical"
)

Found 13340 files belonging to 23 classes.
Using 10672 files for training.
Using 2668 files for validation.


I0000 00:00:1776266437.820248  236607 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 3891 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3050 Ti Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


## Xception

In [24]:
xception_base_model = keras.applications.Xception(
    include_top=False,
    weights="imagenet",
    input_shape=image_size + (3,)
)

xception_base_model.trainable = False # freeze the base model layers

In [26]:
xception_pre_trained_model = keras.models.Sequential()

xception_pre_trained_model.add(layers.Input(shape=image_size + (3,)))
xception_pre_trained_model.add(layers.RandAugment())
xception_pre_trained_model.add(layers.Lambda(keras.applications.xception.preprocess_input)) # normalize to [-1, 1] range that is expected by Xception model

xception_pre_trained_model.add(xception_base_model)

xception_pre_trained_model.add(layers.GlobalAveragePooling2D()) # following the Xception architecture
xception_pre_trained_model.add(layers.Dense(n_classes, activation="softmax"))

xception_pre_trained_model.summary()

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rand_augment_6 (RandAugment)    │ (None, 512, 512, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda_1 (Lambda)               │ (None, 512, 512, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ xception (Functional)           │ (None, 16, 16, 2048)   │    20,861,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 23)             │        47,127 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 20,908,607 (79.76 MB)

 Trainable params: 47,127 (184.09 KB)

 Non-trainable params: 20,861,480 (79.58 MB)

In [27]:
xception_pre_trained_model.compile(
    optimizer=keras.optimizers.Adam(),
    loss=keras.losses.CategoricalCrossentropy(),
    metrics=[
        keras.metrics.CategoricalAccuracy(),
        keras.metrics.F1Score(average='macro'),
        keras.metrics.AUC(multi_label=True, num_labels=n_classes)
    ]
)

In [ ]:
xception_pre_trained_model_history = xception_pre_trained_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=keras.callbacks.EarlyStopping(monitor="val_f1_score", restore_best_weights=True)
)

Epoch 1/10


1334/1334 ━━━━━━━━━━━━━━━━━━━━ 696s 512ms/step - auc_4: 0.8756 - categorical_accuracy: 0.4416 - f1_score: 0.3900 - loss: 1.9691 - val_auc_4: 0.9420 - val_categorical_accuracy: 0.5836 - val_f1_score: 0.5379 - val_loss: 1.4645
Epoch 2/10
1334/1334 ━━━━━━━━━━━━━━━━━━━━ 638s 478ms/step - auc_4: 0.9254 - categorical_accuracy: 0.5612 - f1_score: 0.5273 - loss: 1.5267 - val_auc_4: 0.9543 - val_categorical_accuracy: 0.6409 - val_f1_score: 0.5990 - val_loss: 1.2667
Epoch 3/10
1334/1334 ━━━━━━━━━━━━━━━━━━━━ 577s 433ms/step - auc_4: 0.9365 - categorical_accuracy: 0.5945 - f1_score: 0.5641 - loss: 1.4020 - val_auc_4: 0.9597 - val_categorical_accuracy: 0.6679 - val_f1_score: 0.6300 - val_loss: 1.1666
Epoch 4/10
1334/1334 ━━━━━━━━━━━━━━━━━━━━ 568s 426ms/step - auc_4: 0.9423 - categorical_accuracy: 0.6104 - f1_score: 0.5811 - loss: 1.3298 - val_auc_4: 0.9616 - val_categorical_accuracy: 0.6822 - val_f1_score: 0.6519 - val_loss: 1.1193
Epoch 5/10
1334/1334 ━━━━━━━━━━━━━━━━━━━━ 562s 422ms/step - auc_4: 

: 

## VGG16

In [13]:
vgg16_base_model = keras.applications.VGG16(
    include_top=False,
    weights="imagenet",
    input_shape=image_size + (3,),
    classes=n_classes
)

vgg16_base_model.trainable = False

In [15]:
vgg16_pre_trained_model = keras.models.Sequential()

vgg16_pre_trained_model.add(layers.Input(shape=image_size + (3,)))
vgg16_pre_trained_model.add(layers.RandAugment())

vgg16_pre_trained_model.add(vgg16_base_model)

vgg16_pre_trained_model.add(layers.Flatten()) # following the VGG16 architecture
vgg16_pre_trained_model.add(layers.Dense(n_classes, activation="softmax"))

vgg16_pre_trained_model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rand_augment_2 (RandAugment)    │ (None, 512, 512, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ vgg16 (Functional)              │ (None, 16, 16, 512)    │    14,714,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 131072)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 23)             │     3,014,679 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 17,729,367 (67.63 MB)

 Trainable params: 3,014,679 (11.50 MB)

 Non-trainable params: 14,714,688 (56.13 MB)

In [16]:
vgg16_pre_trained_model.compile(
    optimizer=keras.optimizers.Adam(),
    loss=keras.losses.CategoricalCrossentropy(),
    metrics=[
        keras.metrics.CategoricalAccuracy(),
        keras.metrics.F1Score(average='macro'),
        keras.metrics.AUC(multi_label=True, num_labels=n_classes)
    ]
)

In [ ]:
vgg16_pre_trained_model_history = vgg16_pre_trained_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=keras.callbacks.EarlyStopping(monitor="val_f1_score", restore_best_weights=True)
)

Epoch 1/10


2026-04-12 15:34:19.615306: W external/local_xla/xla/tsl/framework/bfc_allocator.cc:310] Allocator (GPU_0_bfc) ran out of memory trying to allocate 1.02GiB with freed_by_count=0. The caller indicates that this is not a failure, but this may mean that there could be performance gains if more memory were available.
2026-04-12 15:34:19.670071: W external/local_xla/xla/tsl/framework/bfc_allocator.cc:310] Allocator (GPU_0_bfc) ran out of memory trying to allocate 1.02GiB with freed_by_count=0. The caller indicates that this is not a failure, but this may mean that there could be performance gains if more memory were available.
2026-04-12 15:34:19.813237: W external/local_xla/xla/tsl/framework/bfc_allocator.cc:310] Allocator (GPU_0_bfc) ran out of memory trying to allocate 1.02GiB with freed_by_count=0. The caller indicates that this is not a failure, but this may mean that there could be performance gains if more memory were available.
2026-04-12 15:34:19.813311: W external/local_xla/xla/ts

1334/1334 ━━━━━━━━━━━━━━━━━━━━ 853s 631ms/step - auc_1: 0.6633 - categorical_accuracy: 0.3748 - f1_score: 0.3474 - loss: 123.9469 - val_auc_1: 0.7573 - val_categorical_accuracy: 0.5562 - val_f1_score: 0.5162 - val_loss: 108.8529
Epoch 2/10
1334/1334 ━━━━━━━━━━━━━━━━━━━━ 822s 616ms/step - auc_1: 0.7441 - categorical_accuracy: 0.5280 - f1_score: 0.5045 - loss: 128.2739 - val_auc_1: 0.7621 - val_categorical_accuracy: 0.5521 - val_f1_score: 0.5224 - val_loss: 135.4183
Epoch 3/10
1334/1334 ━━━━━━━━━━━━━━━━━━━━ 823s 617ms/step - auc_1: 0.7754 - categorical_accuracy: 0.5833 - f1_score: 0.5643 - loss: 123.9547 - val_auc_1: 0.7650 - val_categorical_accuracy: 0.5746 - val_f1_score: 0.5408 - val_loss: 163.9958
Epoch 4/10
1334/1334 ━━━━━━━━━━━━━━━━━━━━ 826s 619ms/step - auc_1: 0.7949 - categorical_accuracy: 0.6213 - f1_score: 0.6031 - loss: 121.0981 - val_auc_1: 0.7685 - val_categorical_accuracy: 0.5675 - val_f1_score: 0.5400 - val_loss: 187.5976
Epoch 5/10
1334/1334 ━━━━━━━━━━━━━━━━━━━━ 826s 619m

KeyboardInterrupt: 

## VGG19

In [18]:
vgg19_base_model = keras.applications.VGG19(
    include_top=False,
    weights="imagenet",
    input_shape=image_size + (3,),
    classes=n_classes
)

vgg19_base_model.trainable = False

80134624/80134624 ━━━━━━━━━━━━━━━━━━━━ 11s 0us/step


In [20]:
vgg19_pre_trained_model = keras.models.Sequential()

vgg19_pre_trained_model.add(layers.Input(shape=image_size + (3,)))
vgg19_pre_trained_model.add(layers.RandAugment())
vgg19_pre_trained_model.add(layers.Lambda(keras.applications.vgg19.preprocess_input)) # use lambda layer to pass a function

vgg19_pre_trained_model.add(vgg19_base_model)

vgg19_pre_trained_model.add(layers.Flatten()) # following the VGG19 architecture
vgg19_pre_trained_model.add(layers.Dense(n_classes, activation="softmax"))

vgg19_pre_trained_model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rand_augment_4 (RandAugment)    │ (None, 512, 512, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda (Lambda)                 │ (None, 512, 512, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ vgg19 (Functional)              │ (None, 16, 16, 512)    │    20,024,384 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 131072)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 23)             │     3,014,679 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,039,063 (87.89 MB)

 Trainable params: 3,014,679 (11.50 MB)

 Non-trainable params: 20,024,384 (76.39 MB)

In [22]:
vgg19_pre_trained_model.compile(
    optimizer=keras.optimizers.Adam(),
    loss=keras.losses.CategoricalCrossentropy(),
    metrics=[
        keras.metrics.CategoricalAccuracy(),
        keras.metrics.F1Score(average='macro'),
        keras.metrics.AUC(multi_label=True, num_labels=n_classes)
    ]
)

In [ ]:
vgg19_pre_trained_model_history = vgg19_pre_trained_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=keras.callbacks.EarlyStopping(monitor="val_f1_score", restore_best_weights=True)
)

Epoch 1/10


1334/1334 ━━━━━━━━━━━━━━━━━━━━ 1003s 749ms/step - auc_3: 0.6694 - categorical_accuracy: 0.3848 - f1_score: 0.3549 - loss: 100.6082 - val_auc_3: 0.7678 - val_categorical_accuracy: 0.5731 - val_f1_score: 0.5330 - val_loss: 88.6781
Epoch 2/10
1334/1334 ━━━━━━━━━━━━━━━━━━━━ 1002s 751ms/step - auc_3: 0.7510 - categorical_accuracy: 0.5365 - f1_score: 0.5162 - loss: 103.2460 - val_auc_3: 0.7897 - val_categorical_accuracy: 0.6016 - val_f1_score: 0.5704 - val_loss: 99.6278
Epoch 3/10
1334/1334 ━━━━━━━━━━━━━━━━━━━━ 1003s 752ms/step - auc_3: 0.7861 - categorical_accuracy: 0.6018 - f1_score: 0.5845 - loss: 98.5316 - val_auc_3: 0.7856 - val_categorical_accuracy: 0.6211 - val_f1_score: 0.5815 - val_loss: 114.2142
Epoch 4/10
1334/1334 ━━━━━━━━━━━━━━━━━━━━ 1086s 814ms/step - auc_3: 0.8070 - categorical_accuracy: 0.6376 - f1_score: 0.6255 - loss: 99.7441 - val_auc_3: 0.7877 - val_categorical_accuracy: 0.6012 - val_f1_score: 0.5666 - val_loss: 130.2977


## ResNet

We will try the best performing ResNet model available in Keras, ResNet152V2

In [5]:
# resnet_base_model = keras.applications.ResNet152V2(
#     include_top=False,
#     weights="imagenet",
#     input_shape=image_size + (3,),
#     classes=n_classes
# )

# resnet_base_model.trainable = False

In [6]:
# resnet_pre_trained_model = keras.models.Sequential()

# resnet_pre_trained_model.add(layers.Input(shape=image_size + (3,)))
# # resnet_pre_trained_model.add(layers.RandAugment())
# resnet_pre_trained_model.add(layers.Lambda(keras.applications.resnet_v2.preprocess_input)) # use lambda layer to pass a function

# resnet_pre_trained_model.add(resnet_base_model)

# resnet_pre_trained_model.add(layers.GlobalAveragePooling2D())
# resnet_pre_trained_model.add(layers.Dense(n_classes, activation="softmax"))

# resnet_pre_trained_model.summary()

In [5]:
class ResNetPretrained(keras.models.Model):
    
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.preprocessing_layer = layers.Lambda(keras.applications.resnet_v2.preprocess_input)
        self.base_model = keras.applications.ResNet152V2(
            include_top=False,
            weights="imagenet",
            input_shape=image_size + (3,),
            classes=n_classes
        )
        self.base_model.trainable = False
        self.global_avg_pooling_layer = layers.GlobalAveragePooling2D()
        self.dense_layer = layers.Dense(n_classes, activation="softmax")
    
    def call(self, inputs, training=False):
        x = self.preprocessing_layer(inputs)
        
        # The base model contains batchnorm layers. We want to keep them in inference mode
        # when we unfreeze the base model for fine-tuning, so we make sure that the
        # base_model is running in inference mode here.
        x = self.base_model(x, training=False)
        x = self.global_avg_pooling_layer(x)
        outputs = self.dense_layer(x)
        
        return outputs

In [6]:
# class AugmentationScheduler(keras.callba)

In [7]:
resnet_pre_trained_model = ResNetPretrained()

In [8]:
resnet_pre_trained_model.summary()

Model: "res_net_pretrained"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lambda (Lambda)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resnet152v2 (Functional)        │ (None, 16, 16, 2048)   │    58,331,648 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ ?                      │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 58,331,648 (222.52 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 58,331,648 (222.52 MB)

In [9]:
resnet_pre_trained_model.compile(
    optimizer=keras.optimizers.Adam(),
    loss=keras.losses.CategoricalCrossentropy(),
    metrics=[
        keras.metrics.CategoricalAccuracy(),
        keras.metrics.F1Score(average='macro')
    ]
)

In [10]:
# resnet_pre_trained_model_history = resnet_pre_trained_model.fit(
#     train_ds,
#     validation_data=val_ds,
#     batch_size=None,
#     epochs=1,
#     # callbacks=keras.callbacks.EarlyStopping(monitor="val_f1_score", restore_best_weights=True)
# )

In [11]:
resnet_pre_trained_model.base_model.trainable = True

In [12]:
resnet_pre_trained_model.summary()

Model: "res_net_pretrained"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lambda (Lambda)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resnet152v2 (Functional)        │ (None, 16, 16, 2048)   │    58,331,648 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ ?                      │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 58,331,648 (222.52 MB)

 Trainable params: 58,187,904 (221.97 MB)

 Non-trainable params: 143,744 (561.50 KB)

In [13]:
resnet_pre_trained_model.compile(
    optimizer=keras.optimizers.Adam(1e-5), # much lower LR
    loss=keras.losses.CategoricalCrossentropy(),
    metrics=[
        keras.metrics.CategoricalAccuracy(),
        keras.metrics.F1Score(average='macro')
    ]
)

In [14]:
resnet_pre_trained_model_history2 = resnet_pre_trained_model.fit(
    train_ds,
    validation_data=val_ds,
    # initial_epoch=1,
    epochs=1,
    # callbacks=keras.callbacks.EarlyStopping(monitor="val_f1_score", restore_best_weights=True)
)

I0000 00:00:1776087638.437639  283561 service.cc:152] XLA service 0x77b330002d00 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1776087638.437725  283561 service.cc:160]   StreamExecutor device (0): NVIDIA GeForce RTX 3050 Ti Laptop GPU, Compute Capability 8.6
2026-04-13 14:40:40.056163: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1776087647.588620  283561 cuda_dnn.cc:529] Loaded cuDNN version 91002
2026-04-13 14:40:50.941924: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_22664', 132 bytes spill stores, 132 bytes spill loads

2026-04-13 14:40:51.156543: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm

UnknownError: Graph execution error:

Detected at node StatefulPartitionedCall defined at (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main

  File "<frozen runpy>", line 88, in _run_code

  File "/root/miniforge3/envs/DL2526/lib/python3.11/site-packages/ipykernel_launcher.py", line 18, in <module>

  File "/root/miniforge3/envs/DL2526/lib/python3.11/site-packages/traitlets/config/application.py", line 1075, in launch_instance

  File "/root/miniforge3/envs/DL2526/lib/python3.11/site-packages/ipykernel/kernelapp.py", line 758, in start

  File "/root/miniforge3/envs/DL2526/lib/python3.11/site-packages/tornado/platform/asyncio.py", line 211, in start

  File "/root/miniforge3/envs/DL2526/lib/python3.11/asyncio/base_events.py", line 608, in run_forever

  File "/root/miniforge3/envs/DL2526/lib/python3.11/asyncio/base_events.py", line 1936, in _run_once

  File "/root/miniforge3/envs/DL2526/lib/python3.11/asyncio/events.py", line 84, in _run

  File "/root

Not using RandAugment gave us way better results?? Maybe RandAugment is not really suitable for our problem

## MobileNet

Now let's try MobileNetV3Large, which has only 5.4 million parameters, despite being the largest model of the family

In [5]:
class MobileNetV3Pretrained(keras.models.Model):
    
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.base_model = keras.applications.MobileNetV3Large(
            include_top=False,
            include_preprocessing=True, # MobileNetV3 has preprocessing included inside the model
            weights="imagenet",
            input_shape=image_size + (3,),
            classes=n_classes
        )
        self.base_model.trainable = False
        self.global_avg_pooling_layer = layers.GlobalAveragePooling2D()
        self.dense_layer = layers.Dense(n_classes, activation="softmax")
    
    def call(self, inputs, training=False):        
        # The base model contains batchnorm layers. We want to keep them in inference mode
        # when we unfreeze the base model for fine-tuning, so we make sure that the
        # base_model is running in inference mode here.
        x = self.base_model(inputs, training=False)
        x = self.global_avg_pooling_layer(x)
        outputs = self.dense_layer(x)
        
        return outputs

In [17]:
mobilenet_pre_trained_model = MobileNetV3Pretrained()

/root/miniforge3/envs/DL2526/lib/python3.11/site-packages/keras/src/applications/mobilenet_v3.py:519: UserWarning: `input_shape` is undefined or non-square, or `rows` is not 224. Weights for input shape (224, 224) will be loaded as the default.
  return MobileNetV3(


In [18]:
# inputs = keras.Input(shape=image_size + (3,))
# _ = mobilenet_pre_trained_model.call(inputs)
mobilenet_pre_trained_model.summary()

Model: "mobile_net_v3_pretrained_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ MobileNetV3Large (Functional)   │ (None, 16, 16, 960)    │     2,996,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ ?                      │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,996,352 (11.43 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 2,996,352 (11.43 MB)

In [20]:
mobilenet_pre_trained_model.compile(
    optimizer=keras.optimizers.Adam(),
    loss=keras.losses.CategoricalCrossentropy(),
    metrics=[
        keras.metrics.CategoricalAccuracy(),
        keras.metrics.F1Score(average='macro')
    ]
)

In [21]:
mobilenet_pre_trained_model_history = mobilenet_pre_trained_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,
    callbacks=keras.callbacks.EarlyStopping(monitor="val_f1_score", restore_best_weights=True)
)

Epoch 1/5


2026-04-13 23:04:42.575126: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-13 23:04:42.724263: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


2668/2668 ━━━━━━━━━━━━━━━━━━━━ 86s 23ms/step - categorical_accuracy: 0.6468 - f1_score: 0.6202 - loss: 1.2308 - val_categorical_accuracy: 0.7507 - val_f1_score: 0.7234 - val_loss: 0.8500
Epoch 2/5
2668/2668 ━━━━━━━━━━━━━━━━━━━━ 51s 19ms/step - categorical_accuracy: 0.7934 - f1_score: 0.7795 - loss: 0.6965 - val_categorical_accuracy: 0.7654 - val_f1_score: 0.7384 - val_loss: 0.7863
Epoch 3/5
2668/2668 ━━━━━━━━━━━━━━━━━━━━ 56s 21ms/step - categorical_accuracy: 0.8383 - f1_score: 0.8296 - loss: 0.5478 - val_categorical_accuracy: 0.7807 - val_f1_score: 0.7627 - val_loss: 0.7413
Epoch 4/5
2668/2668 ━━━━━━━━━━━━━━━━━━━━ 51s 19ms/step - categorical_accuracy: 0.8651 - f1_score: 0.8578 - loss: 0.4511 - val_categorical_accuracy: 0.7935 - val_f1_score: 0.7728 - val_loss: 0.7186
Epoch 5/5
2668/2668 ━━━━━━━━━━━━━━━━━━━━ 56s 21ms/step - categorical_accuracy: 0.8853 - f1_score: 0.8807 - loss: 0.3833 - val_categorical_accuracy: 0.7852 - val_f1_score: 0.7618 - val_loss: 0.7708


In [22]:
mobilenet_pre_trained_model.base_model.trainable = True

In [23]:
mobilenet_pre_trained_model.compile(
    optimizer=keras.optimizers.SGD(learning_rate=1e-5), # use much lower learning rate for fine-tuning and SGD to save memory
    loss=keras.losses.CategoricalCrossentropy(),
    metrics=[
        keras.metrics.CategoricalAccuracy(),
        keras.metrics.F1Score(average='macro')
    ]
)

In [24]:
mobilenet_pre_trained_model_history = mobilenet_pre_trained_model.fit(
    train_ds,
    validation_data=val_ds,
    initial_epoch=5,
    epochs=10,
    callbacks=keras.callbacks.EarlyStopping(monitor="val_f1_score", restore_best_weights=True)
)

Epoch 6/10
2668/2668 ━━━━━━━━━━━━━━━━━━━━ 164s 37ms/step - categorical_accuracy: 0.9120 - f1_score: 0.9077 - loss: 0.3090 - val_categorical_accuracy: 0.8100 - val_f1_score: 0.7931 - val_loss: 0.6501
Epoch 7/10
2668/2668 ━━━━━━━━━━━━━━━━━━━━ 93s 35ms/step - categorical_accuracy: 0.9187 - f1_score: 0.9155 - loss: 0.2870 - val_categorical_accuracy: 0.8096 - val_f1_score: 0.7917 - val_loss: 0.6410


## EfficientNet

We will use EfficientNetV2S as it seems to be the best tradeoff between performance and parameters that still is usable in our GPU locally

EfficientNetV2S seems to be too big for us to train with batch size 4 and a full resolution image

In [5]:
class EfficientNetV2B2Pretrained(keras.models.Model):
    
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.base_model = keras.applications.EfficientNetV2B2(
            include_top=False,
            include_preprocessing=True, # EfficientNetV2 has preprocessing included inside the model
            weights="imagenet",
            input_shape=image_size + (3,),
            classes=n_classes
        )
        self.base_model.trainable = False
        self.global_avg_pooling_layer = layers.GlobalAveragePooling2D()
        self.dropout_layer = layers.Dropout(0.2)
        self.dense_layer = layers.Dense(n_classes, activation="softmax")
    
    def call(self, inputs, training=False):        
        # The base model contains batchnorm layers. We want to keep them in inference mode
        # when we unfreeze the base model for fine-tuning, so we make sure that the
        # base_model is running in inference mode here.
        x = self.base_model(inputs, training=False)
        x = self.global_avg_pooling_layer(x)
        x = self.dropout_layer(x)
        outputs = self.dense_layer(x)
        
        return outputs

In [6]:
efficientnet_pre_trained_model = EfficientNetV2B2Pretrained()

35839040/35839040 ━━━━━━━━━━━━━━━━━━━━ 125s 4us/step


In [14]:
inputs = keras.Input(shape=image_size + (3,))
_ = efficientnet_pre_trained_model.call(inputs)
efficientnet_pre_trained_model.summary()

Model: "efficient_net_v2b2_pretrained"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ efficientnetv2-b2 (Functional)  │ (None, 16, 16, 1408)   │     8,769,374 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1408)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1408)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 23)             │        32,407 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,801,787 (33.58 MB)

 Trainable params: 8,719,493 (33.26 MB)

 Non-trainable params: 82,288 (321.44 KB)

 Optimizer params: 6 (36.00 B)

In [7]:
efficientnet_pre_trained_model.compile(
    optimizer=keras.optimizers.Adam(),
    loss=keras.losses.CategoricalCrossentropy(),
    metrics=[
        keras.metrics.CategoricalAccuracy(),
        keras.metrics.F1Score(average='macro')
    ]
)

In [8]:
efficientnet_pre_trained_model_history = efficientnet_pre_trained_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,
    callbacks=keras.callbacks.EarlyStopping(monitor="val_f1_score", restore_best_weights=True)
)

Epoch 1/5


I0000 00:00:1776163202.274188    2440 service.cc:152] XLA service 0x7759e4003f40 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1776163202.274291    2440 service.cc:160]   StreamExecutor device (0): NVIDIA GeForce RTX 3050 Ti Laptop GPU, Compute Capability 8.6
2026-04-14 11:40:02.747207: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1776163204.499547    2440 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1776163216.098320    2440 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


2667/2668 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - categorical_accuracy: 0.5525 - f1_score: 0.5144 - loss: 1.5902

2026-04-14 11:41:21.660064: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_3313', 248 bytes spill stores, 248 bytes spill loads

2026-04-14 11:41:21.703012: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_3313', 408 bytes spill stores, 408 bytes spill loads



2668/2668 ━━━━━━━━━━━━━━━━━━━━ 113s 31ms/step - categorical_accuracy: 0.6632 - f1_score: 0.6377 - loss: 1.1878 - val_categorical_accuracy: 0.7504 - val_f1_score: 0.7241 - val_loss: 0.8450
Epoch 2/5
2668/2668 ━━━━━━━━━━━━━━━━━━━━ 75s 28ms/step - categorical_accuracy: 0.7980 - f1_score: 0.7841 - loss: 0.6980 - val_categorical_accuracy: 0.7714 - val_f1_score: 0.7444 - val_loss: 0.7589
Epoch 3/5
2668/2668 ━━━━━━━━━━━━━━━━━━━━ 72s 27ms/step - categorical_accuracy: 0.8400 - f1_score: 0.8297 - loss: 0.5519 - val_categorical_accuracy: 0.7867 - val_f1_score: 0.7621 - val_loss: 0.7145
Epoch 4/5
2668/2668 ━━━━━━━━━━━━━━━━━━━━ 76s 28ms/step - categorical_accuracy: 0.8683 - f1_score: 0.8608 - loss: 0.4589 - val_categorical_accuracy: 0.7924 - val_f1_score: 0.7701 - val_loss: 0.6941
Epoch 5/5
2668/2668 ━━━━━━━━━━━━━━━━━━━━ 72s 27ms/step - categorical_accuracy: 0.8894 - f1_score: 0.8845 - loss: 0.3919 - val_categorical_accuracy: 0.8043 - val_f1_score: 0.7846 - val_loss: 0.6822


In [11]:
efficientnet_pre_trained_model.base_model.trainable = True

In [12]:
efficientnet_pre_trained_model.compile(
    optimizer=keras.optimizers.SGD(learning_rate=1e-5), # use much lower learning rate for fine-tuning and SGD to save memory
    loss=keras.losses.CategoricalCrossentropy(),
    metrics=[
        keras.metrics.CategoricalAccuracy(),
        keras.metrics.F1Score(average='macro')
    ]
)

In [13]:
efficientnet_pre_trained_model_history2 = efficientnet_pre_trained_model.fit(
    train_ds,
    validation_data=val_ds,
    initial_epoch=5,
    epochs=10,
    callbacks=keras.callbacks.EarlyStopping(monitor="val_f1_score", restore_best_weights=True)
)

Epoch 6/10


2026-04-14 11:50:38.919218: E external/local_xla/xla/service/slow_operation_alarm.cc:73] Trying algorithm eng0{} for conv %cudnn-conv-bw-filter.84 = (f16[16,3,3,16]{3,2,1,0}, u8[0]{0}) custom-call(f16[4,256,256,16]{3,2,1,0} %bitcast.58896, f16[4,256,256,16]{3,2,1,0} %bitcast.58898), window={size=3x3 pad=1_1x1_1}, dim_labels=b01f_o01i->b01f, custom_call_target="__cudnn$convBackwardFilter", metadata={op_type="Conv2DBackpropFilter" op_name="gradient_tape/efficient_net_v2b2_pretrained_1/efficientnetv2-b2_1/block1b_project_conv_1/convolution/Conv2DBackpropFilter" source_file="/root/miniforge3/envs/DL2526/lib/python3.11/site-packages/tensorflow/python/framework/ops.py" source_line=1200}, backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"cudnn_conv_backend_config":{"activation_mode":"kNone","conv_result_scale":1,"side_input_scale":0,"leakyrelu_alpha":0},"force_earliest_schedule":false} is taking a while...
2026-04-14 11:50:39.068676: E external/local_xla/xla/service/slow

2668/2668 ━━━━━━━━━━━━━━━━━━━━ 263s 68ms/step - categorical_accuracy: 0.9190 - f1_score: 0.9158 - loss: 0.3051 - val_categorical_accuracy: 0.8145 - val_f1_score: 0.7926 - val_loss: 0.6322
Epoch 7/10
2668/2668 ━━━━━━━━━━━━━━━━━━━━ 170s 64ms/step - categorical_accuracy: 0.9247 - f1_score: 0.9216 - loss: 0.2854 - val_categorical_accuracy: 0.8141 - val_f1_score: 0.7936 - val_loss: 0.6244
Epoch 8/10
2668/2668 ━━━━━━━━━━━━━━━━━━━━ 166s 62ms/step - categorical_accuracy: 0.9267 - f1_score: 0.9236 - loss: 0.2771 - val_categorical_accuracy: 0.8160 - val_f1_score: 0.7948 - val_loss: 0.6198
Epoch 9/10
2668/2668 ━━━━━━━━━━━━━━━━━━━━ 166s 62ms/step - categorical_accuracy: 0.9282 - f1_score: 0.9254 - loss: 0.2708 - val_categorical_accuracy: 0.8171 - val_f1_score: 0.7959 - val_loss: 0.6165
Epoch 10/10
2668/2668 ━━━━━━━━━━━━━━━━━━━━ 167s 62ms/step - categorical_accuracy: 0.9291 - f1_score: 0.9263 - loss: 0.2655 - val_categorical_accuracy: 0.8175 - val_f1_score: 0.7960 - val_loss: 0.6137


## ConvNeXt - Tiny with finetuning

We'll now try ConvNeXtTiny, the smallest model in the family, having 28 million parameters, almost 3x the size of EfficientNetB2

Maybe we should try also ConvNeXtBase, as this was the main model developed, and the other ones are scaled versions that seem to have a worse tradeoff between number of parameters and ImageNet performance

In [ ]:
class ConvNeXtTinyPretrained(keras.models.Model):
    
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.base_model = keras.applications.ConvNeXtTiny(
            include_top=False,
            include_preprocessing=True, # ConvNeXt has preprocessing included inside the model
            weights="imagenet",
            input_shape=image_size + (3,),
            classes=n_classes
        )
        self.base_model.trainable = False
        self.global_avg_pooling_layer = layers.GlobalAveragePooling2D()
        self.dropout_layer = layers.Dropout(0.2)
        self.dense_layer = layers.Dense(n_classes, activation="softmax")
    
    def call(self, inputs, training=False):        
        # The base model contains batchnorm layers. We want to keep them in inference mode
        # when we unfreeze the base model for fine-tuning, so we make sure that the
        # base_model is running in inference mode here.
        x = self.base_model(inputs, training=False)
        x = self.global_avg_pooling_layer(x)
        x = self.dropout_layer(x)
        outputs = self.dense_layer(x)
        
        return outputs

In [10]:
convnext_pre_trained_model = ConvNeXtTinyPretrained()

111650432/111650432 ━━━━━━━━━━━━━━━━━━━━ 83s 1us/step


In [16]:
inputs = keras.Input(shape=image_size + (3,))
_ = convnext_pre_trained_model.call(inputs)
convnext_pre_trained_model.summary()

Model: "conv_ne_xt_tiny_pretrained"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ convnext_tiny (Functional)      │ (None, 16, 16, 768)    │    27,820,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 768)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 768)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 23)             │        17,687 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 27,837,815 (106.19 MB)

 Trainable params: 17,687 (69.09 KB)

 Non-trainable params: 27,820,128 (106.13 MB)

In [18]:
convnext_pre_trained_model.compile(
    optimizer=keras.optimizers.Adam(),
    loss=keras.losses.CategoricalCrossentropy(),
    metrics=[
        keras.metrics.CategoricalAccuracy(),
        keras.metrics.F1Score(average='macro')
    ]
)

In [19]:
convnext_pre_trained_model_history = convnext_pre_trained_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,
    callbacks=keras.callbacks.EarlyStopping(monitor="val_f1_score", restore_best_weights=True)
)

Epoch 1/5


2026-04-14 12:12:34.040701: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot', 16 bytes spill stores, 16 bytes spill loads

2026-04-14 12:12:34.089863: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot', 324 bytes spill stores, 324 bytes spill loads

2026-04-14 12:12:34.090133: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot', 16 bytes spill stores, 16 bytes spill loads

2026-04-14 12:12:35.043613: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_1', 380 bytes spill stores, 360 bytes spill loads

2026-04-14 12:12:35.440016: I external/local_xla/xla/s

2668/2668 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - categorical_accuracy: 0.4739 - f1_score: 0.4235 - loss: 1.9319

2026-04-14 12:14:59.198238: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_3607', 432 bytes spill stores, 432 bytes spill loads



2668/2668 ━━━━━━━━━━━━━━━━━━━━ 198s 63ms/step - categorical_accuracy: 0.6046 - f1_score: 0.5763 - loss: 1.4747 - val_categorical_accuracy: 0.7088 - val_f1_score: 0.6828 - val_loss: 1.0576
Epoch 2/5
2668/2668 ━━━━━━━━━━━━━━━━━━━━ 158s 59ms/step - categorical_accuracy: 0.7642 - f1_score: 0.7491 - loss: 0.8688 - val_categorical_accuracy: 0.7526 - val_f1_score: 0.7279 - val_loss: 0.8737
Epoch 3/5
2668/2668 ━━━━━━━━━━━━━━━━━━━━ 158s 59ms/step - categorical_accuracy: 0.8094 - f1_score: 0.7986 - loss: 0.6980 - val_categorical_accuracy: 0.7740 - val_f1_score: 0.7505 - val_loss: 0.7925
Epoch 4/5
2668/2668 ━━━━━━━━━━━━━━━━━━━━ 158s 59ms/step - categorical_accuracy: 0.8377 - f1_score: 0.8302 - loss: 0.5975 - val_categorical_accuracy: 0.7819 - val_f1_score: 0.7580 - val_loss: 0.7445
Epoch 5/5
2668/2668 ━━━━━━━━━━━━━━━━━━━━ 157s 59ms/step - categorical_accuracy: 0.8565 - f1_score: 0.8511 - loss: 0.5276 - val_categorical_accuracy: 0.7879 - val_f1_score: 0.7654 - val_loss: 0.7171


Should we try to fine-tune the whole model? I don't know if our GPU will have capacity for that, alternatively we could try fine-tuning only the deepest layers, as these are the ones that have the bigger receptive field and capture the most general patterns

It fits!

In [20]:
convnext_pre_trained_model.base_model.trainable = True

In [21]:
convnext_pre_trained_model.compile(
    optimizer=keras.optimizers.SGD(learning_rate=1e-5), # use much lower learning rate for fine-tuning and SGD to save memory
    loss=keras.losses.CategoricalCrossentropy(),
    metrics=[
        keras.metrics.CategoricalAccuracy(),
        keras.metrics.F1Score(average='macro')
    ]
)

In [22]:
convnext_pre_trained_model_history2 = convnext_pre_trained_model.fit(
    train_ds,
    validation_data=val_ds,
    initial_epoch=5,
    epochs=10,
    callbacks=keras.callbacks.EarlyStopping(monitor="val_f1_score", restore_best_weights=True)
)

Epoch 6/10


2026-04-14 12:29:18.511392: E external/local_xla/xla/service/slow_operation_alarm.cc:73] Trying algorithm eng0{} for conv %cudnn-conv-bw-filter.26 = (f16[192,2,2,96]{3,2,1,0}, u8[0]{0}) custom-call(f16[4,128,128,96]{3,2,1,0} %bitcast.37228, f16[4,64,64,192]{3,2,1,0} %bitcast.37249), window={size=2x2 stride=2x2}, dim_labels=b01f_o01i->b01f, custom_call_target="__cudnn$convBackwardFilter", metadata={op_type="Conv2DBackpropFilter" op_name="gradient_tape/conv_ne_xt_tiny_pretrained_1/convnext_tiny_1/convnext_tiny_downsampling_block_0_1/convnext_tiny_downsampling_conv_0_1/convolution/Conv2DBackpropFilter" source_file="/root/miniforge3/envs/DL2526/lib/python3.11/site-packages/tensorflow/python/framework/ops.py" source_line=1200}, backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"cudnn_conv_backend_config":{"activation_mode":"kNone","conv_result_scale":1,"side_input_scale":0,"leakyrelu_alpha":0},"force_earliest_schedule":false} is taking a while...
2026-04-14 12:29:20.099

2668/2668 ━━━━━━━━━━━━━━━━━━━━ 681s 228ms/step - categorical_accuracy: 0.8822 - f1_score: 0.8776 - loss: 0.4476 - val_categorical_accuracy: 0.7939 - val_f1_score: 0.7710 - val_loss: 0.6892
Epoch 7/10
2668/2668 ━━━━━━━━━━━━━━━━━━━━ 612s 229ms/step - categorical_accuracy: 0.8851 - f1_score: 0.8805 - loss: 0.4313 - val_categorical_accuracy: 0.7954 - val_f1_score: 0.7713 - val_loss: 0.6819
Epoch 8/10
2668/2668 ━━━━━━━━━━━━━━━━━━━━ 612s 229ms/step - categorical_accuracy: 0.8876 - f1_score: 0.8832 - loss: 0.4232 - val_categorical_accuracy: 0.7987 - val_f1_score: 0.7760 - val_loss: 0.6774
Epoch 9/10
2668/2668 ━━━━━━━━━━━━━━━━━━━━ 615s 231ms/step - categorical_accuracy: 0.8901 - f1_score: 0.8857 - loss: 0.4176 - val_categorical_accuracy: 0.7991 - val_f1_score: 0.7764 - val_loss: 0.6745
Epoch 10/10
2668/2668 ━━━━━━━━━━━━━━━━━━━━ 619s 232ms/step - categorical_accuracy: 0.8909 - f1_score: 0.8867 - loss: 0.4128 - val_categorical_accuracy: 0.7995 - val_f1_score: 0.7774 - val_loss: 0.6714


## ConvNeXt - Large without finetuning

We'll now try ConvNeXtTiny, the smallest model in the family, having 28 million parameters, almost 3x the size of EfficientNetB2

Maybe we should try also ConvNeXtBase, as this was the main model developed, and the other ones are scaled versions that seem to have a worse tradeoff between number of parameters and ImageNet performance

In [5]:
class ConvNeXtLargePretrained(keras.models.Model):
    
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.base_model = keras.applications.ConvNeXtLarge(
            include_top=False,
            include_preprocessing=True, # ConvNeXt has preprocessing included inside the model
            weights="imagenet",
            input_shape=image_size + (3,),
            classes=n_classes
        )
        self.base_model.trainable = False
        self.global_avg_pooling_layer = layers.GlobalAveragePooling2D()
        self.dropout_layer = layers.Dropout(0.2)
        self.dense_layer = layers.Dense(n_classes, activation="softmax")
    
    def call(self, inputs, training=False):        
        # The base model contains batchnorm layers. We want to keep them in inference mode
        # when we unfreeze the base model for fine-tuning, so we make sure that the
        # base_model is running in inference mode here.
        x = self.base_model(inputs, training=False)
        x = self.global_avg_pooling_layer(x)
        x = self.dropout_layer(x)
        outputs = self.dense_layer(x)
        
        return outputs

In [6]:
convnext_large_pre_trained_model = ConvNeXtLargePretrained()

A local file was found, but it seems to be incomplete or outdated because the auto file hash does not match the original value of 96f02b6f0753d4f543261bc9d09bed650f24dd6bc02ddde3066135b63d23a1cd so we will re-download the data.
785596384/785596384 ━━━━━━━━━━━━━━━━━━━━ 68s 0us/step


In [7]:
inputs = keras.Input(shape=image_size + (3,))
_ = convnext_large_pre_trained_model.call(inputs)
convnext_large_pre_trained_model.summary()

Model: "conv_ne_xt_large_pretrained"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ convnext_large (Functional)     │ (None, 16, 16, 1536)   │   196,230,336 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1536)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1536)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 23)             │        35,351 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 196,265,687 (748.69 MB)

 Trainable params: 35,351 (138.09 KB)

 Non-trainable params: 196,230,336 (748.56 MB)

In [8]:
convnext_large_pre_trained_model.compile(
    optimizer=keras.optimizers.Adam(),
    loss=keras.losses.CategoricalCrossentropy(),
    metrics=[
        keras.metrics.CategoricalAccuracy(),
        keras.metrics.F1Score(average='macro')
    ]
)

In [9]:
convnext_large_pre_trained_model_history = convnext_large_pre_trained_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,
    callbacks=keras.callbacks.EarlyStopping(monitor="val_f1_score", restore_best_weights=True)
)

Epoch 1/5


I0000 00:00:1776169965.710690   55315 service.cc:152] XLA service 0x7da850004150 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1776169965.710749   55315 service.cc:160]   StreamExecutor device (0): NVIDIA GeForce RTX 3050 Ti Laptop GPU, Compute Capability 8.6
2026-04-14 13:32:46.196727: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1776169968.317267   55315 cuda_dnn.cc:529] Loaded cuDNN version 91002
2026-04-14 13:32:49.982318: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot', 356 bytes spill stores, 356 bytes spill loads

2026-04-14 13:32:50.314652: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusio

2668/2668 ━━━━━━━━━━━━━━━━━━━━ 0s 217ms/step - categorical_accuracy: 0.6039 - f1_score: 0.5736 - loss: 1.3870

2026-04-14 13:42:52.887228: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_6847', 144 bytes spill stores, 144 bytes spill loads



2668/2668 ━━━━━━━━━━━━━━━━━━━━ 777s 279ms/step - categorical_accuracy: 0.7198 - f1_score: 0.7010 - loss: 0.9689 - val_categorical_accuracy: 0.8148 - val_f1_score: 0.7963 - val_loss: 0.6397
Epoch 2/5
2668/2668 ━━━━━━━━━━━━━━━━━━━━ 725s 272ms/step - categorical_accuracy: 0.8688 - f1_score: 0.8610 - loss: 0.4572 - val_categorical_accuracy: 0.8265 - val_f1_score: 0.8097 - val_loss: 0.5626
Epoch 3/5
2668/2668 ━━━━━━━━━━━━━━━━━━━━ 725s 272ms/step - categorical_accuracy: 0.9165 - f1_score: 0.9133 - loss: 0.3062 - val_categorical_accuracy: 0.8471 - val_f1_score: 0.8314 - val_loss: 0.5029
Epoch 4/5
2668/2668 ━━━━━━━━━━━━━━━━━━━━ 724s 271ms/step - categorical_accuracy: 0.9450 - f1_score: 0.9439 - loss: 0.2186 - val_categorical_accuracy: 0.8448 - val_f1_score: 0.8263 - val_loss: 0.5048


Now let's try using flatten instead of GlobalAveragePooling before the classification head, and see if it improves or worsens the scores since it has more parameters

In [15]:
class ConvNeXtLarge2Pretrained(keras.models.Model):
    
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.base_model = keras.applications.ConvNeXtLarge(
            include_top=False,
            include_preprocessing=True, # ConvNeXt has preprocessing included inside the model
            weights="imagenet",
            input_shape=image_size + (3,),
            classes=n_classes
        )
        self.base_model.trainable = False
        self.flatten_layer = layers.Flatten()
        self.dropout_layer = layers.Dropout(0.2)
        self.dense_layer = layers.Dense(n_classes, activation="softmax")
    
    def call(self, inputs, training=False):        
        # The base model contains batchnorm layers. We want to keep them in inference mode
        # when we unfreeze the base model for fine-tuning, so we make sure that the
        # base_model is running in inference mode here.
        x = self.base_model(inputs, training=False)
        x = self.flatten_layer(x)
        x = self.dropout_layer(x)
        outputs = self.dense_layer(x)
        
        return outputs

In [16]:
convnext_large2_pre_trained_model = ConvNeXtLarge2Pretrained()

In [17]:
inputs = keras.Input(shape=image_size + (3,))
_ = convnext_large2_pre_trained_model.call(inputs)
convnext_large2_pre_trained_model.summary()

Model: "conv_ne_xt_large2_pretrained_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ convnext_large (Functional)     │ (None, 16, 16, 1536)   │   196,230,336 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 393216)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 393216)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 23)             │     9,043,991 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 205,274,327 (783.06 MB)

 Trainable params: 9,043,991 (34.50 MB)

 Non-trainable params: 196,230,336 (748.56 MB)

In [ ]:
convnext_large2_pre_trained_model.compile(
    optimizer=keras.optimizers.Adam(1e-4), # use lower learning rate because we have a lot more parameters
    loss=keras.losses.CategoricalCrossentropy(),
    metrics=[
        keras.metrics.CategoricalAccuracy(),
        keras.metrics.F1Score(average='macro')
    ]
)

In [19]:
convnext_large2_pre_trained_model_history = convnext_large2_pre_trained_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,
    callbacks=keras.callbacks.EarlyStopping(monitor="val_f1_score", restore_best_weights=True)
)

Epoch 1/5
2668/2668 ━━━━━━━━━━━━━━━━━━━━ 0s 227ms/step - categorical_accuracy: 0.2597 - f1_score: 0.1735 - loss: 10.9251

: 

Kernel crashed, possibly because of the bigger amount of parameters

Let's also try GlobalMaxPooling, which is better for identifying local patterns and could be useful in our problem of identifying paintings

In [5]:
class ConvNeXtLarge3Pretrained(keras.models.Model):
    
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.base_model = keras.applications.ConvNeXtLarge(
            include_top=False,
            include_preprocessing=True, # ConvNeXt has preprocessing included inside the model
            weights="imagenet",
            input_shape=image_size + (3,),
            classes=n_classes,
            pooling="max"
        )
        self.base_model.trainable = False
        # self.global_max_pooling_layer = layers.GlobalMaxPooling2D()
        self.dropout_layer = layers.Dropout(0.2)
        self.dense_layer = layers.Dense(n_classes, activation="softmax")
    
    def call(self, inputs, training=False):        
        # The base model contains batchnorm layers. We want to keep them in inference mode
        # when we unfreeze the base model for fine-tuning, so we make sure that the
        # base_model is running in inference mode here.
        x = self.base_model(inputs, training=False)
        # x = self.global_max_pooling_layer(x)
        x = self.dropout_layer(x)
        outputs = self.dense_layer(x)
        
        return outputs

In [6]:
convnext_large3_pre_trained_model = ConvNeXtLarge3Pretrained()

In [23]:
inputs = keras.Input(shape=image_size + (3,))
_ = convnext_large3_pre_trained_model.call(inputs)
convnext_large3_pre_trained_model.summary()

Model: "conv_ne_xt_large3_pretrained_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ convnext_large (Functional)     │ (None, 1536)           │   196,230,336 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 1536)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 23)             │        35,351 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 196,265,687 (748.69 MB)

 Trainable params: 35,351 (138.09 KB)

 Non-trainable params: 196,230,336 (748.56 MB)

In [7]:
convnext_large3_pre_trained_model.compile(
    optimizer=keras.optimizers.Adam(),
    loss=keras.losses.CategoricalCrossentropy(),
    metrics=[
        keras.metrics.CategoricalAccuracy(),
        keras.metrics.F1Score(average='macro')
    ]
)

In [8]:
convnext_large3_pre_trained_model_history = convnext_large3_pre_trained_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,
    callbacks=keras.callbacks.EarlyStopping(monitor="val_f1_score", restore_best_weights=True)
)

Epoch 1/5


I0000 00:00:1776178157.122122  107845 service.cc:152] XLA service 0x729848003d60 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1776178157.122203  107845 service.cc:160]   StreamExecutor device (0): NVIDIA GeForce RTX 3050 Ti Laptop GPU, Compute Capability 8.6
2026-04-14 15:49:17.623004: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1776178159.752881  107845 cuda_dnn.cc:529] Loaded cuDNN version 91002
2026-04-14 15:49:21.413826: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot', 356 bytes spill stores, 356 bytes spill loads

2026-04-14 15:49:21.538017: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusio

2668/2668 ━━━━━━━━━━━━━━━━━━━━ 0s 220ms/step - categorical_accuracy: 0.4977 - f1_score: 0.4642 - loss: 1.8245

2026-04-14 15:59:32.309613: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_6839', 144 bytes spill stores, 144 bytes spill loads



2668/2668 ━━━━━━━━━━━━━━━━━━━━ 787s 281ms/step - categorical_accuracy: 0.6089 - f1_score: 0.5833 - loss: 1.3629 - val_categorical_accuracy: 0.6893 - val_f1_score: 0.6681 - val_loss: 1.0637
Epoch 2/5
2668/2668 ━━━━━━━━━━━━━━━━━━━━ 743s 279ms/step - categorical_accuracy: 0.8470 - f1_score: 0.8439 - loss: 0.5117 - val_categorical_accuracy: 0.7001 - val_f1_score: 0.6791 - val_loss: 1.1135
Epoch 3/5
2668/2668 ━━━━━━━━━━━━━━━━━━━━ 735s 275ms/step - categorical_accuracy: 0.9259 - f1_score: 0.9255 - loss: 0.2700 - val_categorical_accuracy: 0.6949 - val_f1_score: 0.6772 - val_loss: 1.2264


In [ ]:
# convnext_large3_pre_trained_model.save("../outputs/models/convnext_large_max_pre_trained.keras")

## The next approach may be to use the largest model possible and finetuning only a subset of layers (maybe only the deepest), so that it fits in the VRAM